# *Linear* Kelvin–Voigt Airway-Tree **Newton** Solver with Array-Based Tree Traversal

## 0. Explanation:

Use the **tree topology** directly: information is first condensed from the terminal units toward the root and then recovered from the root back to the terminal branches.

*Main characteristics of this version:*

- **_Topology preprocessing_**: the code first builds parent-child relations, identifies the root airway, separates airways and terminal units, and groups the airway elements into bottom-up and top-down layers.

- **_NO topology-based ordering_**: the initial order comes from `element_ids = list(elements.keys())`. This normally **follows the element order in the loaded JSON** dictionary.

- **_Array-based solver data_**: after the **readable dictionary** representation is built, the tree is **converted into NumPy arrays** using global element indices. This reduces dependence on Python objects during the solver step.

- **_Struct-of-Arrays style_**: physical and numerical quantities such as `R`, `G`, `h`, `Pin`, `Pout`, `Q`, and terminal-unit volumes are **stored in separate arrays**. This makes the data layout closer to what would later be useful in a C/C++ implementation.

- **_0D terminal-unit model_**: each terminal unit has one internal pressure, `P_TU`, in addition to `P_pleural`. The outlet pressure of the connected terminal airway is equal to `P_TU`; there is no separate TU inlet/outlet pressure or TU resistance equation.

- **_Newton solver_**: Every **Newton correction/step** contains a bottom-up and a top-down traversal.

- **_Layer-based bottom-up and top-down passes_**: the solver avoids recursion and instead processes the tree layer by layer. The bottom-up pass computes the local subtree relation `dQ = G dPin + h`; the top-down pass recovers pressure, flow, and terminal-unit **corrections**.

- **_Computational behavior_**: each time step mainly consists of two tree traversals. Therefore, the expected cost grows approximately linearly with the number of elements, while preprocessing and JSON conversion are separated from the repeated solve.

- **_Prototype focus_**: this version is partly NumPy-oriented and array-based, but the core tree traversal still uses explicit Python loops over layers and elements.

## 1. Data loading:
Data loaded from **JSON**: 

- `node_coordinates`: *Not used* geometry information.
- `element_nodes`: *Used* by `build_elements()` to determine parent-child connectivity.
- `element_type`: *Used* to assign `"Airway"` or `"TerminalUnit"` to every element.
- `generation`: *Used* by `build_element_layer()`.
- `radius`: *Not used*, for AW resistance.
- `TU Volume`: *Used* to initialize TU reference, old, and new volumes.

In [157]:
import json
import numpy as np
from collections import defaultdict

def load_simple_lung_json(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)
    return data["node_coordinates"], data["element_nodes"], data["element_type"], data["generation"], data["radius"], data["volume"]

def load_gen10_json(file_path):
    with open(file_path, "r") as f:
        # Main data dictionary:
        data = json.load(f)

    node_coordinates = data["nodepos"]
    element_nodes = {}
    element_type = {}
    generation = {}
    radius = {}
    volume = {}

    # Element classification: AW or TU?
    for element_id, airway in data["airways"].items():
        element_nodes[element_id] = airway["nodeids"]
        element_type[element_id] = "Airway"
        generation[element_id] = airway["generation"] - 1
        radius[element_id] = airway["radius"]

    for element_id, cluster in data["alveolarclusters"].items():
        element_nodes[element_id] = cluster["nodeids"]
        element_type[element_id] = "TerminalUnit"
        generation[element_id] = -1
        radius[element_id] = cluster["radius"]
        volume[element_id] = cluster["volume"]

    return node_coordinates, element_nodes, element_type, generation, radius, volume

### Select Input file:

In [158]:
input_format = "gen10"
#input_format = "simple"

if input_format == "simple":
    node_coordinates, element_nodes, element_type, generation, radius, volume = load_simple_lung_json("reduced_lung_3_aw_2_tu_fields_with_V.json")
elif input_format == "gen10":
    node_coordinates, element_nodes, element_type, generation, radius, volume = load_gen10_json("lung_tree_gen10__Notation_template.json")
else:
    raise ValueError(f"Unknown input_format: {input_format}")

## 2. Preprocessing:

### Build elements dictionary:
Data stored in **`elements`**: 

- `type`: *Used* to identify TU (from JSON).
- `parent`: *Used* by `find_root_element()` to identify single root element.
- `children`: *Used* by `compile_tree_to_soa` to construct the `child_index`arrays for layers.
- `is_terminal`: *Used* to identify *leaf elements* in the layers..
- `resistance`: *Used* for the SoA AW resistance array. It is zero for TUs.

In [159]:
# Build element dictionaries:
def build_elements(element_nodes, element_type):
    
    # Using as reference the starting and ending node:
    start_node = {element_id: nodes[0] for element_id, nodes in element_nodes.items()}
    end_node = {element_id: nodes[-1] for element_id, nodes in element_nodes.items()}
    elements_starting_at_node = defaultdict(list)

    for element_id, node_id in start_node.items():
        elements_starting_at_node[node_id].append(element_id)

    # Detect children
    children_by_element = defaultdict(list)
    parent_by_element = {}
    for element_id, node_id in end_node.items():
        for child_id in elements_starting_at_node.get(node_id, []):
            children_by_element[element_id].append(child_id)
            parent_by_element[child_id] = element_id

    return {
        element_id: {
            "type": element_type[element_id],       # Only used when initializing TUs
            "parent": parent_by_element.get(element_id),
            "children": children_by_element[element_id],
            "is_terminal": len(children_by_element[element_id]) == 0,
            "resistance": 1.0 if element_type[element_id] == 'Airway' else 0.0,
        }
        for element_id in element_nodes
    }

# Find root element:
def find_root_element(elements):
    roots = [element_id for element_id, element in elements.items() if element["parent"] is None]
    if len(roots) != 1:
        raise ValueError(f"Expected exactly one root element, found {roots}")
    return roots[0]

elements = build_elements(element_nodes, element_type)
root_element_id = find_root_element(elements)

### Build bottom-up layers:

```python
bottom_up_layers = [
    ["AW1", "AW2"],                 # Deepest airways connected to the TUs
    ["AW0"]]                # Root airway

```

In [160]:
def build_airway_layers(elements, generation):
    """
    Build bottom-up layers containing only airway IDs.
    Terminal units are handled separately.
    """
    airway_layers = defaultdict(list)

    for element_id, element in elements.items():
        # Keep only AW and groups them by generation
        if element["type"] == "Airway":
            airway_layers[generation[element_id]].append(element_id)
    
    # Sort generation from deepest to root
    return [airway_layers[gen] 
            for gen in sorted(airway_layers, reverse=True)]

bottom_up_aw_layers = build_airway_layers(elements, generation)

### Physical parameters, boundary conditions, Newton settings, and initial TU state
Terminal Units are initialized here!

In [161]:
dt = 0.1
root_pressure = 1.0

# Newton settings
newton_max_iterations = 30
newton_residual_tolerance = 1.0e-10         # F(x^(k))

def pleural_pressure(time):
    return -2.0 * np.sin(2.0 * np.pi * time)

# Initialize terminal-unit state:
terminal_units = {
    element_id: {
        "V0": volume[element_id],
        "E": 8.0,
        "eta": 2.0,
        "V_old": volume[element_id],
        "V_new": volume[element_id],
        "P_pleural": 0.0,
    }
    for element_id, element in elements.items()
    if element["type"] == "TerminalUnit"
}

## 3. Compile tree into array-based solver data:

Converts the readable tree structure (dictionaries) into a solver friendly structure based on integer indices and Numpy arrays. **`elements`** is **not used anymore**.
    
    ==> Bottom-up and Top-down solver can be faster!

```python
solver_data = {
    # Tree topology and indexing
    "root_airway_index": ...,
    "bottom_up_layers": ..., 
    "top_down_layers": ..., 
    "airway_children": ..., 
    "airway_to_tu": ...,
    
    # Airway arrays
    "airways": { 
        "R": ..., 
        "Pin": ..., 
        "Pout": ..., 
        "Q": ..., 
        "G": ..., 
        "h": ..., 
        "dPin": ..., 
        "dPout": ..., 
        "dQ": ..., 
        },

    # Terminal-unit arrays 
    "terminal_units": { 
        "P": ..., 
        "Q": ..., 
        "G": ..., 
        "h": ..., 
        "dP": ..., 
        "dQ": ..., 
        "V0": ...,          # Unchanged over time
        "V_old": ..., 
        "V_new": ..., 
        "E": ..., 
        "eta": ..., 
        "P_pleural": ..., 
        }, 
    }
```

In [ ]:
def compile_tree_to_soa(elements, terminal_units, bottom_up_aw_layers, root_element_id):
    """
    Compile the tree into separate airway and TU Struct-of-Arrays.
    No global numerical element indexing is used.
    """
    # ---------------------------------------------------------
    # Separate airway and TU IDs
    # ---------------------------------------------------------
    aw_ids = [element_id for element_id, element in elements.items()
              if element["type"] == "Airway"]
    tu_ids = list(terminal_units.keys())

    element_to_aw_index = {element_id: aw_i
        for aw_i, element_id in enumerate(aw_ids)}
    element_to_tu_index = {element_id: tu_i
        for tu_i, element_id in enumerate(tu_ids)}

    n_aw = len(aw_ids)
    n_tu = len(tu_ids)

    root_aw_index = element_to_aw_index[root_element_id]

    # ---------------------------------------------------------
    # Typed child-index tables
    # ---------------------------------------------------------
    # Child index for each AW
    aw_child_aw_index = -np.ones((n_aw, 2), dtype=int)
    aw_child_tu_index = -np.ones((n_aw, 2), dtype=int)

    # For each AW:
    for aw_id in aw_ids:
        # Local array index of aw_id
        aw_i = element_to_aw_index[aw_id]
        # Children of aw_id
        children = elements[aw_id]["children"]

        if len(children) > 2:
            raise ValueError(f"Airway {aw_id} has more than two children.")

        for child_slot, child_id in enumerate(children):
            # If child is AW:
            if child_id in element_to_aw_index:
                # Store local AW index of that child:
                aw_child_aw_index[aw_i, child_slot] = element_to_aw_index[child_id]
            # If child is a TU:
            elif child_id in element_to_tu_index:
                # Store local AW index of that child:
                aw_child_tu_index[aw_i, child_slot] = element_to_tu_index[child_id]

            else:
                raise ValueError(f"Unknown child {child_id} of airway {aw_id}.")

    # ---------------------------------------------------------
    # Convert airway layers from IDs to airway indices
    # ---------------------------------------------------------
    aw_layers = [np.array(
        [element_to_aw_index[element_id] for element_id in layer], dtype=int)
        for layer in bottom_up_aw_layers
        ]

    # ---------------------------------------------------------
    # Create separated Struct-of-Arrays
    # ---------------------------------------------------------
    solver_data = {
        # Optional metadata for connecting results to JSON IDs
        "aw_ids": aw_ids,
        "tu_ids": tu_ids,

        # Airway topology
        "root_aw_index": root_aw_index,
        "aw_layers": aw_layers,
        "aw_child_aw_index": aw_child_aw_index,
        "aw_child_tu_index": aw_child_tu_index,

        # Airway physical and solver arrays
        "aw_R": np.array([elements[element_id]["resistance"] for element_id in aw_ids], dtype=float),
        "aw_G": np.zeros(n_aw),
        "aw_h": np.zeros(n_aw),
        "aw_Pin": np.zeros(n_aw),
        "aw_Pout": np.zeros(n_aw),
        "aw_Q": np.zeros(n_aw),

        # Airway Newton increments
        "aw_dPin": np.zeros(n_aw),
        "aw_dPout": np.zeros(n_aw),
        "aw_dQ": np.zeros(n_aw),

        # TU solver arrays
        "tu_G": np.zeros(n_tu),
        "tu_h": np.zeros(n_tu),
        "tu_P": np.zeros(n_tu),
        "tu_Q": np.zeros(n_tu),

        # TU Newton increments
        "tu_dP": np.zeros(n_tu),
        "tu_dQ": np.zeros(n_tu),

        # TU physical state
        "tu_V0": np.zeros(n_tu),
        "tu_V_old": np.zeros(n_tu),
        "tu_V_new": np.zeros(n_tu),
        "tu_E": np.zeros(n_tu),
        "tu_eta": np.zeros(n_tu),
        "tu_P_pleural": np.zeros(n_tu),

        # TU Newton work arrays
        "tu_residual": np.zeros(n_tu),
        "tu_tangent": np.zeros(n_tu),
    }

    # ---------------------------------------------------------
    # Store TU physical data
    # ---------------------------------------------------------
    for element_id, unit in terminal_units.items():
        tu_i = element_to_tu_index[element_id]

        solver_data["tu_V0"][tu_i] = unit["V0"] 
        solver_data["tu_V_old"][tu_i] = unit["V_old"]
        solver_data["tu_V_new"][tu_i] = unit["V_new"]
        solver_data["tu_E"][tu_i] = unit["E"]
        solver_data["tu_eta"][tu_i] = unit["eta"]
        solver_data["tu_P_pleural"][tu_i] = unit["P_pleural"]

    return solver_data

solver_data = compile_tree_to_soa(elements, terminal_units, bottom_up_aw_layers, root_element_id,)

### Newton residual evaluation
The function evaluates the **current equation errors F(xᵏ)**. It returns one normalized residual measure for the convergence check and stores terminal-unit trial volumes, residuals, and tangents.

In [163]:
def evaluate_nonlinear_residuals(solver_data):
    """
    Evaluate the separated airway and TU equations.
    """
    aw_R = solver_data["aw_R"]
    aw_Pin = solver_data["aw_Pin"]
    aw_Pout = solver_data["aw_Pout"]
    aw_Q = solver_data["aw_Q"]

    tu_P = solver_data["tu_P"]
    tu_Q = solver_data["tu_Q"]

    pressure_residuals = []
    flow_residuals = []

    # ---------------------------------------------------------
    # Terminal-unit Kelvin-Voigt residual
    # ---------------------------------------------------------
    # V_trial = V_old + dt * Q_TU
    V_trial = solver_data["tu_V_old"] + dt * tu_Q

    if np.any(V_trial <= 0.0):
        raise RuntimeError("Non-positive terminal trial volume.")

    # F_terminal:
    F_terminal = (solver_data["tu_P_pleural"] + solver_data["tu_E"] * (V_trial - solver_data["tu_V0"])
        / solver_data["tu_V0"] + solver_data["tu_eta"] * tu_Q / solver_data["tu_V0"] - tu_P)

    K_elastic = solver_data["tu_E"] / solver_data["tu_V0"]

    solver_data["tu_residual"][:] = F_terminal
    solver_data["tu_tangent"][:] = K_elastic

    if F_terminal.size:
        pressure_residuals.append(np.max(np.abs(F_terminal)))

    # ---------------------------------------------------------
    # Airway equations
    # ---------------------------------------------------------
    n_aw = aw_Q.size

    for aw_i in range(n_aw):
        # F_AW_R:
        F_airway = (aw_Pin[aw_i] - aw_Pout[aw_i] - aw_R[aw_i] * aw_Q[aw_i])
        pressure_residuals.append(abs(F_airway))

        child_flow = 0.0

        # For each child (max 2):
        for child_slot in range(2):
            # Store local index of the child
            child_aw_i = solver_data["aw_child_aw_index"][aw_i, child_slot]
            child_tu_i = solver_data["aw_child_tu_index"][aw_i, child_slot]

            if child_aw_i >= 0:     # If child is AW
                child_pressure = aw_Pin[child_aw_i]
                child_flow += aw_Q[child_aw_i]          # Sum of Q_children

            elif child_tu_i >= 0:   # If child is TU
                child_pressure = tu_P[child_tu_i]
                child_flow += tu_Q[child_tu_i]          # Sum of Q_children

            else:
                continue

            # F_cont = P_out,parent - P_in,child
            F_continuity = (aw_Pout[aw_i] - child_pressure)
            pressure_residuals.append(abs(F_continuity))

        # F_junct = Q_parent - sum(Q_children)
        F_junction = aw_Q[aw_i] - child_flow
        flow_residuals.append(abs(F_junction))

    # Root-pressure boundary condition
    root_aw_i = solver_data["root_aw_index"]

    # F_root = P_in - P_BC
    F_root = aw_Pin[root_aw_i] - root_pressure
    pressure_residuals.append(abs(F_root))

    # ---------------------------------------------------------
    # Normalized convergence measure
    # ---------------------------------------------------------
    # Max F_pressure, F_flow
    max_pressure_residual = max(pressure_residuals, default=0.0)
    max_flow_residual = max(flow_residuals, default=0.0)

    pressure_scale = max(1.0, abs(root_pressure), np.max(np.abs(aw_Pin)), 
                         np.max(np.abs(aw_Pout)), np.max(np.abs(tu_P)), 
                         np.max(np.abs(solver_data["tu_P_pleural"])))
    flow_scale = max(1.0, np.max(np.abs(aw_Q)), np.max(np.abs(tu_Q)))

    # Normalized values:
    normalized_pressure_residual = max_pressure_residual / pressure_scale
    normalized_flow_residual = max_flow_residual / flow_scale

    return {
        "measure": max(normalized_pressure_residual, normalized_flow_residual),
        "terminal": (np.max(np.abs(F_terminal)) if F_terminal.size else 0.0)
    }

## 4. Tree solver:

### Bottom-up pass:
The bottom-up pass condenses the linear Newton correction system

J(xᵏ) Δx = −F(xᵏ).

Each subtree is represented by

ΔQ = **G** ΔPin + **h**.

The terminal-unit relations are computed first, after which the child relations are combined successively toward the root.

In [164]:
def compute_incremental_subtree_relation(solver_data):
    """
    Compute separate TU and airway subtree relations.
    """
    # Clear previous relations
    solver_data["aw_G"][:] = 0.0
    solver_data["aw_h"][:] = 0.0
    solver_data["tu_G"][:] = 0.0
    solver_data["tu_h"][:] = 0.0

    # ---------------------------------------------------------
    # Terminal-unit relations
    # ---------------------------------------------------------
    D_terminal = (solver_data["tu_eta"] / solver_data["tu_V0"] + dt * solver_data["tu_tangent"])

    # CALCULATE G & H FOR TUs:
    solver_data["tu_G"][:] = 1.0 / D_terminal
    solver_data["tu_h"][:] = -solver_data["tu_residual"] / D_terminal

    # ---------------------------------------------------------
    # Airway relations, deepest layer to root
    # ---------------------------------------------------------
    aw_R = solver_data["aw_R"]
    aw_G = solver_data["aw_G"]
    aw_h = solver_data["aw_h"]
    aw_Pin = solver_data["aw_Pin"]
    aw_Pout = solver_data["aw_Pout"]
    aw_Q = solver_data["aw_Q"]

    tu_G = solver_data["tu_G"]
    tu_h = solver_data["tu_h"]
    tu_P = solver_data["tu_P"]
    tu_Q = solver_data["tu_Q"]

    # For each AW layer:
    for layer in solver_data["aw_layers"]:
        for aw_i in layer:
            G_downstream = 0.0
            h_downstream = 0.0
            child_flow = 0.0

            # For each child of each AW (max 2):
            for child_slot in range(2):
                # Recover AW/TU index
                child_aw_i = solver_data["aw_child_aw_index"][aw_i, child_slot]
                child_tu_i = solver_data["aw_child_tu_index"][aw_i, child_slot]

                # If child is AW:
                if child_aw_i >= 0:
                    child_G = aw_G[child_aw_i]
                    child_h = aw_h[child_aw_i]
                    child_pressure = aw_Pin[child_aw_i]
                    child_Q = aw_Q[child_aw_i]

                # If child is TU:
                elif child_tu_i >= 0:
                    child_G = tu_G[child_tu_i]
                    child_h = tu_h[child_tu_i]
                    child_pressure = tu_P[child_tu_i]
                    child_Q = tu_Q[child_tu_i]

                else:
                    continue

                # Still for each child (2):
                # F_cont = P_out,parent - P_in,child 
                F_continuity = aw_Pout[aw_i] - child_pressure

                G_downstream += child_G
                h_downstream += child_h + child_G * F_continuity

                child_flow += child_Q

            # F_AW_R:
            F_airway = (aw_Pin[aw_i] - aw_Pout[aw_i] - aw_R[aw_i] * aw_Q[aw_i])

            # F_junct = Q_parent - sum(Q_child)
            F_junction = aw_Q[aw_i] - child_flow

            denominator = 1.0 + aw_R[aw_i] * G_downstream

            # CALCULATE G & H FOR AWs:
            aw_G[aw_i] = G_downstream / denominator
            aw_h[aw_i] = (h_downstream - F_junction + G_downstream * F_airway) / denominator

### Top-down pass:
The top-down pass starts with the root-pressure correction and uses the
previously computed G and h relations to recover

**ΔPin, ΔPout, and ΔQ**

for every airway and terminal unit.

In [165]:
def recover_pressure_and_flow_increments(solver_data):
    """
    Recover separate AW and TU Newton increments.
    """
    aw_R = solver_data["aw_R"]
    aw_G = solver_data["aw_G"]
    aw_h = solver_data["aw_h"]
    aw_Pin = solver_data["aw_Pin"]
    aw_Pout = solver_data["aw_Pout"]
    aw_Q = solver_data["aw_Q"]

    aw_dPin = solver_data["aw_dPin"]
    aw_dPout = solver_data["aw_dPout"]
    aw_dQ = solver_data["aw_dQ"]

    tu_P = solver_data["tu_P"]
    tu_dP = solver_data["tu_dP"]
    tu_dQ = solver_data["tu_dQ"]

    aw_dPin[:] = 0.0
    aw_dPout[:] = 0.0
    aw_dQ[:] = 0.0
    tu_dP[:] = 0.0
    tu_dQ[:] = 0.0

    # Root-pressure correction
    root_aw_i = solver_data["root_aw_index"]
    aw_dPin[root_aw_i] = root_pressure - aw_Pin[root_aw_i]

    # Root toward terminal units
    for layer in reversed(solver_data["aw_layers"]):
        for aw_i in layer:
            # dQ = G * dPin + h:
            aw_dQ[aw_i] = (aw_G[aw_i] * aw_dPin[aw_i] + aw_h[aw_i])
            
            # F_AW_R:
            F_airway = (aw_Pin[aw_i] - aw_Pout[aw_i] - aw_R[aw_i] * aw_Q[aw_i])
            # dPout = dPin - R*Q + F_AW_R
            aw_dPout[aw_i] = (aw_dPin[aw_i] - aw_R[aw_i] * aw_dQ[aw_i] + F_airway)

            for child_slot in range(2):
                # Recover local index of the child in that slot:
                child_aw_i = solver_data["aw_child_aw_index"][aw_i, child_slot]
                child_tu_i = solver_data["aw_child_tu_index"][aw_i, child_slot]

                # If child is AW:
                if child_aw_i >= 0:
                    #F_cont = Pout,parent - Pin,child
                    F_continuity = (aw_Pout[aw_i] - aw_Pin[child_aw_i])
                    # dPin = dPout + F_cont
                    aw_dPin[child_aw_i] = (aw_dPout[aw_i] + F_continuity)

                # If child is TU:
                elif child_tu_i >= 0:
                    #F_cont = Pout,parent - Pin,child
                    F_continuity = (aw_Pout[aw_i] - tu_P[child_tu_i])
                    # dP_TU = dPout + F_cont
                    tu_dP[child_tu_i] = (aw_dPout[aw_i] + F_continuity)

    # All TU pressure increments dP_TU are now known
    tu_dQ[:] = (solver_data["tu_G"] * tu_dP + solver_data["tu_h"])

### Measure and apply the Newton correction

In [166]:
def apply_newton_increment(solver_data, minimum_damping=1.0e-8):
    """
    Apply separated AW and TU Newton increments.
    """
    damping = 1.0

    while damping >= minimum_damping:
        
        # Q^(k+1) = Q^(k) + lambda*dQ
        aw_Q_candidate = (solver_data["aw_Q"] + damping * solver_data["aw_dQ"])
        tu_Q_candidate = (solver_data["tu_Q"] + damping * solver_data["tu_dQ"])

        # V^(k+1) = V^(k) + dt*Q^(k+1)
        V_candidate = (solver_data["tu_V_old"] + dt * tu_Q_candidate)

        # If all V are positive:
        if np.all(V_candidate > 0.0):

            # x^(k+1) = x^(k) + dx :
            solver_data["aw_Pin"][:] += (damping * solver_data["aw_dPin"])
            solver_data["aw_Pout"][:] += (damping * solver_data["aw_dPout"])

            solver_data["aw_Q"][:] = aw_Q_candidate

            solver_data["tu_P"][:] += (damping * solver_data["tu_dP"])
            solver_data["tu_Q"][:] = tu_Q_candidate

            return damping

        damping *= 0.5

    raise RuntimeError("No positive-volume Newton step could be found.")

## 5. Newton iteration and physical time stepping

In [167]:
# Solve one newton step:
def solve_one_nonlinear_time_step(solver_data, time, verbose=False):
    """
    Solve one implicit physical time step using Newton's method.

    The function name follows the nonlinear reference workflow,
    although the terminal material equation is linear.

    tu_V_old remains fixed throughout all Newton iterations and is
    committed only after convergence.
    """
    # Set pleural pressure once for this physical time step
    solver_data["tu_P_pleural"][:] = pleural_pressure(time)

    # Preserve the old physical-time state during Newton
    V_old_at_start = solver_data["tu_V_old"].copy()

    converged = False
    # Evaluate F(x^k)
    residual_before = evaluate_nonlinear_residuals(solver_data)

    for iteration in range(newton_max_iterations):
        # IGNORE
        if verbose:
            print(
                f"time={time:.6g}, "
                f"iteration={iteration}, "
                f"residual={residual_before['measure']:.6e}, "
                f"terminal={residual_before['terminal']:.6e}"
            )

        # If the initial guess already solver the equations...
        if (residual_before["measure"] < newton_residual_tolerance):
            converged = True
            break

        # Condense and solve J(x^k) dx = -F(x^k) with tree solver!
        compute_incremental_subtree_relation(solver_data)
        recover_pressure_and_flow_increments(solver_data)

        # Apply x^(k+1) = x^k + damping*dx
        damping = apply_newton_increment(solver_data)   # Values are changed here!

        # Evaluate F(x^(k+1))
        residual_after = evaluate_nonlinear_residuals(solver_data)

        if verbose:
            print(
                f"    damping={damping:.6e}, "
                f"new residual={residual_after['measure']:.6e}"
            )

        # If convergence test passed...
        if residual_after["measure"] < newton_residual_tolerance:
            converged = True
            break

        # Reuse F(x^(k+1)) as F(x^k) in the next iteration
        residual_before = residual_after
    
        # If convergence test failed: next k-newton iteration

    # If no convergence and k-max iterations: FAIL
    if not converged:
        final_residual = residual_after

        raise RuntimeError(
            f"Newton did not converge at time={time}. "
            f"Final residual="
            f"{final_residual['measure']:.6e}"
        )

    # V-changed test (optional/safety)
    if not np.array_equal(solver_data["tu_V_old"], V_old_at_start):
        raise RuntimeError("tu_V_old changed during Newton.")

    # New Volume after convergence
    solver_data["tu_V_new"][:] = (solver_data["tu_V_old"] + dt * solver_data["tu_Q"])

    if np.any(solver_data["tu_V_new"] <= 0.0):
        raise ValueError("Converged solution has a non-positive terminal volume.")

    # Commit the converged physical-time state
    solver_data["tu_V_old"][:] = (solver_data["tu_V_new"])

    return iteration + 1

### Orchestration function and Reset:

In [168]:
# Orchestration function for physical time steps:
def solve_time_steps(solver_data, number_of_steps, verbose=False):
    """
    Solve the requested number of physical time steps.

    The original target-notebook convention time = step*dt is
    intentionally preserved.
    """
    newton_iterations_per_step = []

    for step in range(number_of_steps):
        time = step * dt

        iterations = solve_one_nonlinear_time_step(solver_data, time, verbose=verbose)

        newton_iterations_per_step.append(iterations)

    return np.asarray(newton_iterations_per_step)


# Reset values:
def reset_simulation_state(solver_data):
    """
    Restore the initial state before a fresh complete simulation.

    Call this once before solve_time_steps, not inside each physical
    time step.
    """
    for name in ["aw_G","aw_h","aw_Pin","aw_Pout","aw_Q","aw_dPin","aw_dPout","aw_dQ",
                 "tu_G","tu_h","tu_P","tu_Q","tu_dP","tu_dQ"]:
        solver_data[name][:] = 0.0

    solver_data["tu_P_pleural"][:] = 0.0
    solver_data["tu_residual"][:] = 0.0
    solver_data["tu_tangent"][:] = 0.0

    solver_data["tu_V_old"][:] = solver_data["tu_V0"]

    solver_data["tu_V_new"][:] =solver_data["tu_V0"]

## 6. Run the simulation and inspect the results

In [169]:
reset_simulation_state(solver_data)

newton_iterations = solve_time_steps(solver_data, number_of_steps=10, verbose=True)

print("Newton iterations per physical time step:")
print(newton_iterations)

print("Final airway flows:")
print(solver_data["aw_Q"])

print("Final TU flows:")
print(solver_data["tu_Q"])

print("Final TU pressures:")
print(solver_data["tu_P"])

print("Final terminal volumes:")
print(solver_data["tu_V_new"])

time=0, iteration=0, residual=1.000000e+00, terminal=0.000000e+00
    damping=1.000000e+00, new residual=8.301194e-16
time=0.1, iteration=0, residual=9.999995e-01, terminal=1.175570e+00
    damping=1.000000e+00, new residual=6.748380e-14
time=0.2, iteration=0, residual=3.819653e-01, terminal=7.265412e-01
    damping=1.000000e+00, new residual=2.102745e-14
time=0.3, iteration=0, residual=3.279733e-05, terminal=6.238423e-05
    damping=1.000000e+00, new residual=8.171503e-16
time=0.4, iteration=0, residual=3.820005e-01, terminal=7.266044e-01
    damping=1.000000e+00, new residual=3.915576e-14
time=0.5, iteration=0, residual=1.000048e+00, terminal=1.175616e+00
    damping=1.000000e+00, new residual=9.103829e-14
time=0.6, iteration=0, residual=1.000017e+00, terminal=1.175591e+00
    damping=1.000000e+00, new residual=6.616929e-14
time=0.7, iteration=0, residual=3.819660e-01, terminal=7.265424e-01
    damping=1.000000e+00, new residual=4.718448e-14
time=0.8, iteration=0, residual=1.134142e-